In [ ]:
import cdsapi

c = cdsapi.Client(
)

c.retrieve(
    "reanalysis-era5-complete",
    {
        "stream": "oper",
        "type": "an",
        "levtype": "pv",
        "levelist": "2000",
        "param": "3",
        "date": "2026-05-10/to/2026-06-01",
        "time": "00/06/12/18",
        "grid": "0.25/0.25",
        "format": "grib",
    },
    "theta_2pvu_20260510_20260601.grib",
)

In [ ]:
# Download 6-hourly ERA5 mean sea-level pressure (MSLP).
# ECMWF parameter 151 is mean sea-level pressure on the surface level.
c.retrieve(
    "reanalysis-era5-complete",
    {
        "stream": "oper",
        "type": "an",
        "levtype": "sfc",
        "param": "151",
        "date": "2026-05-10/to/2026-06-01",
        "time": "00/06/12/18",
        "grid": "0.25/0.25",
        "format": "grib",
    },
    "mslp_20260510_20260601.grib",
)


In [ ]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/path/to/cdo/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())

In [ ]:
from cdo import Cdo

cdo = Cdo()

base = "./06_derived_data/rwb_era5/"
infile = base + "theta_2pvu_20260510_20260601.grib"
outfile = base + "theta_2pvu_20260510_20260601.nc"

cdo.setgridtype(
    "regular",
    input=infile,
    output=outfile,
    options="-f nc --eccodes"
)

print(f"Converted with ecCodes: {outfile}")

In [ ]:
# Convert the 6-hourly MSLP GRIB file to regular-grid NetCDF.
mslp_infile = base + "mslp_20260510_20260601.grib"
mslp_outfile = base + "mslp_20260510_20260601.nc"

cdo.setgridtype(
    "regular",
    input=mslp_infile,
    output=mslp_outfile,
    options="-f nc --eccodes",
)

print(f"Converted MSLP with ecCodes: {mslp_outfile}")


In [ ]:
from cdo import Cdo
import os

cdo = Cdo()

infile = base + "theta_2pvu_20260510_20260601.grib"
gaussfile = base + "theta_2pvu_20260510_20260601_n320.grib"
outfile = base + "theta_2pvu_20260510_20260601_T45.nc"

# 1) remap to a global Gaussian grid
cdo.remapbil("n320", input=infile, output=gaussfile)

print("Created Gaussian-grid file:", gaussfile, os.path.exists(gaussfile))
print(cdo.sinfo(input=gaussfile))

specfile = base + "theta_2pvu_20260510_20260601_spec.grb"

cdo.gp2sp(
    input=gaussfile,
    output=specfile,
    options="-f grb"
)

print("Created:", specfile, os.path.exists(specfile))
print(cdo.sinfo(input=specfile))

In [ ]:
specfile_t45 = base+"tmp2026_spec_T45.grb"

cdo.sp2sp("45", input=specfile, output=specfile_t45)

print("Created:", specfile_t45, os.path.exists(specfile_t45))
print(cdo.sinfo(input=specfile_t45))

In [ ]:
cdo.sp2gp(
    input=specfile_t45,
    output=outfile,
    options="-f nc --eccodes"
)

In [ ]:
import xarray as xr
import numpy as np
from find_rwb import find_rwb

ds = xr.open_dataset(base + "theta_2pvu_20260510_20260601_T45.nc")

# choose one timestep
da = ds["pt"].isel(time=0, lev=0)   # shape: (lat, lon)

# arrays for find_rwb
tpot = da.values
latitudes = ds["lat"].values
longitudes = ds["lon"].values

# your requested isentropes: 250, 255, ..., 450 K
isentropes = np.arange(250, 455, 5)

contour_groups, DBs, tpot_extended, longitudes_extended = find_rwb(
    tpot,
    isentropes,
    longitudes,
    latitudes,
    min_contours_in_overturning=2,
    max_clustering_distance=1000,
    use_max_width_condition=False,
)

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

from find_rwb import find_rwb

# =========================
# Input files
# =========================
fn_rwb = base + "theta_2pvu_20260510_20260601_T45.nc"
fn_plot = base + "theta_2pvu_20260510_20260601.nc"
fn_mslp = base + "mslp_20260510_20260601.nc"

outdir = base + "2026_theta_2pvu_RWB"
os.makedirs(outdir, exist_ok=True)

start_date = "2026-05-10"
end_date = "2026-06-01"

theta_levels = np.arange(250, 455, 5)

# Plot low-pressure contours only north of 20 degrees N.
mslp_min_lat = 20.0
mslp_low_max_hpa = 1000
mslp_contour_interval_hpa = 4

mslp_levels_hpa = np.arange(
    900,
    mslp_low_max_hpa + mslp_contour_interval_hpa,
    mslp_contour_interval_hpa,
)

min_contours_in_overturning = 2
max_clustering_distance = 1000
use_max_width_condition = False


def get_theta(ds):
    """Select the potential-temperature variable."""
    if "pt" in ds.data_vars:
        da = ds["pt"]
    elif "theta" in ds.data_vars:
        da = ds["theta"]
    else:
        da = ds[list(ds.data_vars)[0]]

    if "lev" in da.dims:
        da = da.isel(lev=0)

    return da


def get_mslp_hpa(ds):
    """Select MSLP, normalize coordinate names, and convert Pa to hPa."""
    candidates = (
        "msl",
        "mslp",
        "prmsl",
        "mean_sea_level_pressure",
    )

    name = next(
        (candidate for candidate in candidates if candidate in ds),
        None,
    )

    da = ds[name] if name else ds[list(ds.data_vars)[0]]

    rename = {}

    if "latitude" in da.dims:
        rename["latitude"] = "lat"

    if "longitude" in da.dims:
        rename["longitude"] = "lon"

    if rename:
        da = da.rename(rename)

    # Remove singleton surface or level dimensions.
    for dim in list(da.dims):
        if dim not in {"time", "lat", "lon"}:
            if da.sizes[dim] != 1:
                raise ValueError(
                    f"Unexpected non-singleton MSLP dimension: "
                    f"{dim}={da.sizes[dim]}"
                )

            da = da.isel({dim: 0})

    units = str(da.attrs.get("units", "")).strip().lower()

    if units in {"pa", "pascal", "pascals"}:
        da = da / 100.0

    elif units not in {
        "hpa",
        "mbar",
        "millibar",
        "millibars",
    }:
        sample = da.isel(
            {dim: 0 for dim in da.dims}
        ).item()

        if sample > 2000:
            da = da / 100.0

    da.attrs["units"] = "hPa"

    return da


def lon360_to_180(lon):
    """Convert longitude from [0, 360) to [-180, 180)."""
    lon = np.asarray(lon)

    return ((lon + 180) % 360) - 180


def plot_contour_with_wrap(
    ax,
    lons,
    lats,
    color,
    lw=2.0,
    alpha=1.0,
):
    """Plot detected contours without lines crossing the dateline."""
    lons_plot = lon360_to_180(lons)
    lats = np.asarray(lats)

    jumps = np.where(
        np.abs(np.diff(lons_plot)) > 180
    )[0]

    start = 0

    for j in jumps:
        ax.plot(
            lons_plot[start:j + 1],
            lats[start:j + 1],
            color=color,
            linewidth=lw,
            alpha=alpha,
            transform=ccrs.PlateCarree(),
            zorder=5,
        )

        start = j + 1

    ax.plot(
        lons_plot[start:],
        lats[start:],
        color=color,
        linewidth=lw,
        alpha=alpha,
        transform=ccrs.PlateCarree(),
        zorder=5,
    )


# =========================
# Open datasets
# =========================
ds_rwb = xr.open_dataset(fn_rwb)
ds_plot = xr.open_dataset(fn_plot)
ds_mslp = xr.open_dataset(fn_mslp)

da_rwb = get_theta(ds_rwb).sel(
    time=slice(start_date, end_date)
)

da_plot = get_theta(ds_plot).sel(
    time=slice(start_date, end_date)
)

da_mslp = get_mslp_hpa(ds_mslp).sel(
    time=slice(start_date, end_date)
)

# Use only timestamps available in all three files.
common_times = np.intersect1d(
    np.intersect1d(
        da_rwb.time.values,
        da_plot.time.values,
    ),
    da_mslp.time.values,
)

# Longitude ordering for the theta dataset.
plot_lon_180 = lon360_to_180(
    da_plot.lon.values
)

plot_sort_idx = np.argsort(plot_lon_180)
plot_lon_sorted = plot_lon_180[plot_sort_idx]

# Longitude ordering for the MSLP dataset.
mslp_lon_180 = lon360_to_180(
    da_mslp.lon.values
)

mslp_sort_idx = np.argsort(mslp_lon_180)
mslp_lon_sorted = mslp_lon_180[mslp_sort_idx]


# =========================
# Loop over timesteps
# =========================
for t in common_times:

    # ---------------------------------------------
    # RWB detection using the lower-resolution T45
    # ---------------------------------------------
    rwb_field = (
        da_rwb
        .sel(time=t)
        .transpose("lat", "lon")
    )

    tpot = rwb_field.values
    latitudes_rwb = rwb_field.lat.values
    longitudes_rwb = rwb_field.lon.values

    (
        contour_groups,
        DBs,
        tpot_extended,
        longitudes_extended,
    ) = find_rwb(
        tpot,
        theta_levels,
        longitudes_rwb,
        latitudes_rwb,
        min_contours_in_overturning=(
            min_contours_in_overturning
        ),
        max_clustering_distance=(
            max_clustering_distance
        ),
        use_max_width_condition=(
            use_max_width_condition
        ),
    )

    # Keep Northern Hemisphere RWB events.
    filtered_groups = []
    filtered_DBs = []

    for group, db in zip(contour_groups, DBs):
        all_lats = np.concatenate(
            [contour[1] for contour in group]
        )

        if np.mean(all_lats) > 0:
            filtered_groups.append(group)
            filtered_DBs.append(db)

    contour_groups = filtered_groups
    DBs = filtered_DBs

    # ---------------------------------------------
    # Full-resolution theta background
    # ---------------------------------------------
    plot_field = (
        da_plot
        .sel(time=t)
        .transpose("lat", "lon")
    )

    plot_values_sorted = (
        plot_field.values[:, plot_sort_idx]
    )

    (
        plot_values_cyclic,
        plot_lon_cyclic,
    ) = add_cyclic_point(
        plot_values_sorted,
        coord=plot_lon_sorted,
        axis=-1,
    )

    # ---------------------------------------------
    # MSLP overlay north of 20 degrees N
    # ---------------------------------------------
    mslp_field = (
        da_mslp
        .sel(time=t)
        .transpose("lat", "lon")
    )

    # Remove all MSLP data south of 20 degrees N.
    mslp_field = mslp_field.where(
        mslp_field.lat >= mslp_min_lat,
        drop=True,
    )

    mslp_values_sorted = (
        mslp_field.values[:, mslp_sort_idx]
    )

    (
        mslp_values_cyclic,
        mslp_lon_cyclic,
    ) = add_cyclic_point(
        mslp_values_sorted,
        coord=mslp_lon_sorted,
        axis=-1,
    )

    # Select low-pressure levels present north of 20 N.
    mslp_min = np.nanmin(mslp_values_cyclic)
    mslp_max = np.nanmax(mslp_values_cyclic)

    available_mslp_levels = mslp_levels_hpa[
        (mslp_levels_hpa >= mslp_min)
        & (mslp_levels_hpa <= mslp_max)
    ]

    # =========================
    # Plot
    # =========================
    fig = plt.figure(figsize=(12, 7))

    ax = plt.axes(
        projection=ccrs.NorthPolarStereo()
    )

    cf = ax.contourf(
        plot_lon_cyclic,
        plot_field.lat.values,
        plot_values_cyclic,
        levels=theta_levels,
        cmap="viridis",
        extend="both",
        transform=ccrs.PlateCarree(),
    )

    ax.contour(
        plot_lon_cyclic,
        plot_field.lat.values,
        plot_values_cyclic,
        levels=theta_levels,
        colors="black",
        alpha=0.5,
        linewidths=0.2,
        transform=ccrs.PlateCarree(),
    )

    # Low-pressure contours at or below 1000 hPa,
    # restricted to latitudes north of 20 degrees N.
    if available_mslp_levels.size:
        mslp_contours = ax.contour(
            mslp_lon_cyclic,
            mslp_field.lat.values,
            mslp_values_cyclic,
            levels=available_mslp_levels,
            colors="white",
            linewidths=0.7,
            alpha=0.8,
            transform=ccrs.PlateCarree(),
            zorder=4,
        )

    # RWB contours detected from the T45 field.
    for group, db in zip(
        contour_groups,
        DBs,
    ):
        color = "red" if db == 1 else "blue"

        for contour in group:
            plot_contour_with_wrap(
                ax,
                contour[0, :],
                contour[1, :],
                color=color,
                alpha=0.7,
                lw=1.0,
            )

    ax.coastlines()

    ax.add_feature(
        cfeature.BORDERS,
        linewidth=0.4,
    )

    ax.add_feature(
        cfeature.LAND,
        facecolor="lightgray",
        alpha=0.3,
    )

    ax.set_extent(
        [-180, 180, 0, 90],
        crs=ccrs.PlateCarree(),
    )

    cbar = plt.colorbar(
        cf,
        ax=ax,
        orientation="vertical",
        pad=0.02,
    )

    cbar.set_label(
        "Potential temperature [K]"
    )

    date_str = np.datetime_as_string(
        t,
        unit="h",
    )

    ax.set_title(
        "Potential temperature at 2 PVU with low-pressure "
        "MSLP contours\n"
        f"{date_str} | RWB detected at T45 | "
        f"events: {len(contour_groups)} | "
        "red: cyclonic, blue: anticyclonic, "
        "white: MSLP <= 1000 hPa north of 20 degrees N"
    )

    safe_date_str = date_str.replace(":", "")

    outfile = os.path.join(
        outdir,
        (
            "theta_2pvu_RWB_MSLP_T45_detection_"
            f"{safe_date_str}.png"
        ),
    )

    plt.savefig(
        outfile,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close()

    print(f"Saved: {outfile}")


ds_rwb.close()
ds_plot.close()
ds_mslp.close()

In [ ]:
import time

t0 = time.time()
import os
print("os:", time.time() - t0)

t0 = time.time()
import numpy as np
print("numpy:", time.time() - t0)

t0 = time.time()
import xarray as xr
print("xarray:", time.time() - t0)

t0 = time.time()
import matplotlib.pyplot as plt
print("matplotlib:", time.time() - t0)

t0 = time.time()
import cartopy.crs as ccrs
print("cartopy.crs:", time.time() - t0)

t0 = time.time()
import cartopy.feature as cfeature
print("cartopy.feature:", time.time() - t0)

t0 = time.time()
from find_rwb import find_rwb
print("find_rwb:", time.time() - t0)